# Non-Time-Resolved Depth-Profile Workflow

This example shows the first step of a non-time-resolved workflow. The notebook first organizes the raw `.xy` spectra inline by core level, copying each file into a per-core-level subfolder, and then loads the spectra into a `trspecfit` project with the package API so the loader can be reused for other datasets.

In [2]:
import re
import shutil
from pathlib import Path

from trspecfit.utils import load_xy_project

## 1. Group raw files by core level

The raw `.xy` files live together in one folder. Here we read the core level from each file's `Region` header, then copy every file into a compact per-core-level subfolder (`C1s`, `O1s`, `N1s`, `Survey`) under `data/`. `VBM` scans are not their own core level: each is copied into the folder of the core-level scan it shares an excitation energy with (nearest by spectrum ID). Originals stay in place and re-running is idempotent.

In [3]:
# Group raw .xy files by core level and copy them into compact subfolders under
# data/. Originals stay in place; re-running is idempotent.
raw_dir = Path.cwd() / "data" / "LserinepH7_500mM"
grouped_root = raw_dir.parent  # copies land in siblings of the raw folder

CORE_LEVEL_RE = re.compile(r"^(?P<element>[A-Z][a-z]?)\s+(?P<orbital>\d[spdf])")
HEADER_LABELS = {
    "region": "Region:",
    "spectrum_id": "Spectrum ID:",
    "excitation": "Excitation Energy:",
}


def read_header(path):
    """Read Region, Spectrum ID and Excitation Energy from an .xy header.

    Only leading ``#`` metadata lines are scanned; parsing stops at the first
    non-comment (data) line.
    """

    found = {}
    with path.open(encoding="utf-8", errors="replace") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line.startswith("#"):
                break
            body = line.lstrip("#").strip()
            for key, label in HEADER_LABELS.items():
                if body.startswith(label):
                    found[key] = body[len(label):].strip()
    return {
        "region": found.get("region", ""),
        "spectrum_id": int(found["spectrum_id"]),
        "excitation": int(found["excitation"]),
    }


def classify(region):
    """Return (kind, folder) for a Region string.

    kind is 'core', 'survey', 'vbm', or 'unknown'; folder is the compact
    destination name (None for VBM/unknown, resolved later).
    """

    if region.startswith("Survey"):
        return "survey", "Survey"
    if region.startswith("VBM"):
        return "vbm", None
    match = CORE_LEVEL_RE.match(region)
    if match is None:
        return "unknown", None
    return "core", match["element"] + match["orbital"]


records = []
for path in sorted(raw_dir.glob("*.xy")):
    header = read_header(path)
    kind, folder = classify(header["region"])
    records.append({"path": path, "kind": kind, "folder": folder, **header})
records.sort(key=lambda r: r["spectrum_id"])

core_scans = [r for r in records if r["kind"] == "core"]


def pair_vbm(vbm):
    """Nearest core-level scan (either direction) with matching excitation.

    Prefers the preceding scan when two candidates are equidistant.
    """

    candidates = [r for r in core_scans if r["excitation"] == vbm["excitation"]]
    if not candidates:
        return None
    return min(
        candidates,
        key=lambda r: (
            abs(r["spectrum_id"] - vbm["spectrum_id"]),
            r["spectrum_id"] > vbm["spectrum_id"],
        ),
    )


copied = 0
for record in records:
    if record["kind"] in {"core", "survey"}:
        dest_folder = record["folder"]
    elif record["kind"] == "vbm":
        partner = pair_vbm(record)
        if partner is None:
            print(
                f"WARNING: unmatched VBM {record['path'].name} "
                f"(ID {record['spectrum_id']}, Exc {record['excitation']}) "
                f"-> VBM_unmatched"
            )
            dest_folder = "VBM_unmatched"
        else:
            dest_folder = partner["folder"]
    else:
        print(
            f"WARNING: unrecognized region '{record['region']}' in "
            f"{record['path'].name} -> VBM_unmatched"
        )
        dest_folder = "VBM_unmatched"

    dest_dir = grouped_root / dest_folder
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(record["path"], dest_dir / record["path"].name)
    copied += 1

print(f"Copied {copied} files into {grouped_root}")

Copied 89 files into c:\Users\EJRiffe\Desktop\tr-Xray spec\time-resolved-spectroscopy-fit\examples\fitting_workflows\05_non_time_resolved_depth_profile\data


## 2. Point the workflow at your data directory

The loader reads the raw spectra directly from the `LserinepH7_500mM/` subfolder (the originals, not the grouped copies). Adapt the filename pattern so it captures the core-level and depth labels from the file names.

In [6]:
core_level = "C1s"
data_dir = Path.cwd() / "data" /core_level


project, files = load_xy_project(
    data_dir,
    project_path=Path.cwd(),
    project_name="depth_profile_example",

)

project.describe()

Config file c:\Users\EJRiffe\Desktop\tr-Xray spec\time-resolved-spectroscopy-fit\examples\fitting_workflows\05_non_time_resolved_depth_profile\project.yaml not found, using defaults
Project
  path:         c:\Users\EJRiffe\Desktop\tr-Xray spec\time-resolved-spectroscopy-fit\examples\fitting_workflows\05_non_time_resolved_depth_profile
  results:      c:\Users\EJRiffe\Desktop\tr-Xray spec\time-resolved-spectroscopy-fit\examples\fitting_workflows\05_non_time_resolved_depth_profile_fits
  name:         depth_profile_example
  config:       defaults (no YAML loaded)


## 3. Inspect the grouped data

Each entry in `files` is a `trspecfit.File` keyed by `(core_level, depth)`, which makes it easy to inspect a single slice or build a fitting workflow around it.

In [9]:
for key, file in files.items():
    print(key, file.data.shape, file.energy.shape)
    # file.describe()
print(len(files), "files loaded")

('L-serine pH 7 500 mM_610_C 1s 800', '') (251,) (251,)
('L-serine pH 7 500 mM_611_VBM', '') (251,) (251,)
('L-serine pH 7 500 mM_617_C 1s 200', '') (251,) (251,)
('L-serine pH 7 500 mM_618_VBM', '') (251,) (251,)
('L-serine pH 7 500 mM_624_C 1s 600', '') (251,) (251,)
('L-serine pH 7 500 mM_625_VBM', '') (251,) (251,)
('L-serine pH 7 500 mM_631_C 1s 100', '') (251,) (251,)
('L-serine pH 7 500 mM_632_VBM', '') (251,) (251,)
('L-serine pH 7 500 mM_638_C 1s 400', '') (251,) (251,)
('L-serine pH 7 500 mM_639_VBM', '') (251,) (251,)
('L-serine pH 7 500 mM_645_C 1s 300', '') (251,) (251,)
('L-serine pH 7 500 mM_646_VBM', '') (251,) (251,)
('L-serine pH 7 500 mM_655_C 1s 150', '') (251,) (251,)
('L-serine pH 7 500 mM_656_VBM', '') (251,) (251,)
('L-serine pH 7 500 mM_657_C 1s 150', '') (251,) (251,)
('L-serine pH 7 500 mM_663_C 1s 250', '') (251,) (251,)
('L-serine pH 7 500 mM_664_VBM', '') (251,) (251,)
('L-serine pH 7 500 mM_672_C 1s 70', '') (251,) (251,)
('L-serine pH 7 500 mM_673_VBM', 